# 第5章: 大規模言語モデル

この章では、大規模言語モデル (LLM; Large Language Model) の利用し、様々なタスクに取り組む。大規模言語モデルをプログラムからAPI経由で呼び出すことを想定しており、そのAPIの利用で費用が発生する可能性があることに留意せよ。

In [11]:
pip install google-generativeai

In [1]:
import google.generativeai as genai
from google.colab import userdata

In [2]:
# Gemini APIを使う場合
api_key = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key = api_key)
model = genai.GenerativeModel('gemini-2.5-flash-lite')

# 例
# prompt = "日本の首都について教えてください。"
# response = model.generate_content(prompt)
# print(response.text)

## Gemma インストール

In [1]:
# 必要なライブラリをインストール
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 42.2 MB/s eta 0:00:00


In [2]:
# Hugging Faceにログイン（アクセストークンを貼り付けます）
from huggingface_hub import login
login()

# モデルを読み込む
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [15]:
model_id = "google/gemma-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)

# メモリを効率化するため4-bit量子化は必須です
quantization_config = BitsAndBytesConfig(load_in_4bit = True)

gemma = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map = "auto",
    quantization_config=quantization_config
)

# プロンプトを作成してモデルに渡す
chat = [
    {"role": "user", "content": "日本の首都について教えてください。"},
]
prompt = tokenizer.apply_chat_template(chat, tokenize = False, add_generation_prompt = True)
inputs = tokenizer.encode(prompt, add_special_tokens = False, return_tensors = "pt")

# モデルに推論を実行させる
outputs = gemma.generate(input_ids = inputs.to(gemma.device), max_new_tokens = 512)
response = tokenizer.decode(outputs[0], skip_special_tokens = True)

# 応答を表示
print(response)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

user
日本の首都について教えてください。
model
日本の首都は東京です。東京は日本の政治・経済・文化の中心地であり、世界的な重要都市です。


## 40. Zero-Shot推論

以下の問題の解答を作成せよ。ただし、解答生成はzero-shot推論とせよ。

```
9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。
```

出典: [令和5年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [日本史AB 問題](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) 日本史B 1 問3

In [9]:
prompt = """
以下の問題に解答してください。

9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。
"""

response = model.generate_content(prompt)
print(response.text)

正しい年代順は、イ→ウ→アです。


* **イ：嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。**  嵯峨天皇の治世は809年から823年。藤原冬嗣は嵯峨天皇の寵臣として活躍しました。

* **ウ：藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。** 承和の変は842年。この変の後、藤原良房（北家）の勢力が強まり、藤原氏内の優位を確立しました。

* **ア：藤原時平は，策謀を用いて菅原道真を政界から追放した。** 菅原道真の左遷は894年。藤原時平は、この事件の中心人物でした。


したがって、イ、ウ、ア の順が年代的に正しいとなります。



## Gemmaを用いる場合

In [8]:
text = """
以下の問題に解答してください。解答はア、イ、ウの順番だけ答えてください。

9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。
"""

chat = [
    {
        "role": "user",
        "content": text},
]
prompt = tokenizer.apply_chat_template(chat, tokenize = False, add_generation_prompt = True)

# 1. モデルに入力するトークンを作成し、その長さを記憶
inputs = tokenizer.encode(prompt, add_special_tokens = False, return_tensors = "pt")
input_token_length = len(inputs[0])

# モデルに推論を実行させる
outputs = gemma.generate(input_ids=inputs.to(gemma.device), max_new_tokens = 512)

# 2. 全体の出力から、新しく生成された部分のトークンだけをスライス
new_tokens = outputs[0][input_token_length:]

# 3. 新しく生成された部分だけをデコードする
model_answer = tokenizer.decode(new_tokens, skip_special_tokens = True)

# モデルの解答だけを表示
print(model_answer)

ア　9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。


## 41. Few-Shot推論

以下の問題と解答を与え、問題40で示した質問の解答をfew-shot推論（この場合は4-shot推論）で生成せよ。

```
日本の近代化に関連するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　府知事・県令からなる地方官会議が設置された。
イ　廃藩置県が実施され，中央から府知事・県令が派遣される体制になった。
ウ　すべての藩主が，天皇に領地と領民を返還した。

解答: ウ→イ→ア
```

出典: [令和5年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [日本史AB 問題](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) 日本史A 1 問8


```
江戸幕府の北方での対外的な緊張について述べた次の文ア～ウを年代の古い順に正しく並べよ。

ア　レザノフが長崎に来航したが，幕府が冷淡な対応をしたため，ロシア船が樺太や択捉島を攻撃した。
イ　ゴローウニンが国後島に上陸し，幕府の役人に捕らえられ抑留された。
ウ　ラクスマンが根室に来航し，漂流民を届けるとともに通商を求めた。

解答: ウ→ア→イ
```

出典: [令和5年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00010.htm) [日本史AB 問題](https://www.mext.go.jp/content/20240523-mxt_syogai02-mext_000031286_03nihonshi.pdf) 日本史B 3 問3

```
中居屋重兵衛の生涯の期間におこったできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　アヘン戦争がおこり，清がイギリスに敗北した。
イ　異国船打払令が出され，外国船を撃退することが命じられた。
ウ　桜田門外の変がおこり，大老の井伊直弼が暗殺された。

解答: イ→ア→ウ
```

出典: [令和4年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00007.htm) [日本史 問題](https://www.mext.go.jp/content/20240513-mxt_syogai02-mext_00002452_03nihonshi.pdf) 日本史A 1 問1


```
加藤高明が外務大臣として提言を行ってから、内閣総理大臣となり演説を行うまでの時期のできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　朝鮮半島において，独立を求める大衆運動である三・一独立運動が展開された。
イ　関東大震災後の混乱のなかで，朝鮮人や中国人に対する殺傷事件がおきた。
ウ　日本政府が，袁世凱政府に対して二十一カ条の要求を突き付けた。

解答: ウ→ア→イ
```

出典: [令和4年度第1回高等学校卒業程度認定試験問題](https://www.mext.go.jp/a_menu/koutou/shiken/kakomon/1411255_00007.htm) [日本史 問題](https://www.mext.go.jp/content/20240513-mxt_syogai02-mext_00002452_03nihonshi.pdf) 日本史A 2 問4


In [10]:
prompt = """
以下の問題に解答してください。

9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。



また、以下の問題を参考にしてください。

日本の近代化に関連するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　府知事・県令からなる地方官会議が設置された。
イ　廃藩置県が実施され，中央から府知事・県令が派遣される体制になった。
ウ　すべての藩主が，天皇に領地と領民を返還した。

解答: ウ→イ→ア

江戸幕府の北方での対外的な緊張について述べた次の文ア～ウを年代の古い順に正しく並べよ。

ア　レザノフが長崎に来航したが，幕府が冷淡な対応をしたため，ロシア船が樺太や択捉島を攻撃した。
イ　ゴローウニンが国後島に上陸し，幕府の役人に捕らえられ抑留された。
ウ　ラクスマンが根室に来航し，漂流民を届けるとともに通商を求めた。

解答: ウ→ア→イ

中居屋重兵衛の生涯の期間におこったできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　アヘン戦争がおこり，清がイギリスに敗北した。
イ　異国船打払令が出され，外国船を撃退することが命じられた。
ウ　桜田門外の変がおこり，大老の井伊直弼が暗殺された。

解答: イ→ア→ウ

加藤高明が外務大臣として提言を行ってから、内閣総理大臣となり演説を行うまでの時期のできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　朝鮮半島において，独立を求める大衆運動である三・一独立運動が展開された。
イ　関東大震災後の混乱のなかで，朝鮮人や中国人に対する殺傷事件がおきた。
ウ　日本政府が，袁世凱政府に対して二十一カ条の要求を突き付けた。

解答: ウ→ア→イ
"""

response = model.generate_content(prompt)
print(response.text)

9世紀の出来事の年代順は、以下のようになります。

**イ→ウ→ア**

理由：

* **イ：嵯峨天皇の蔵人頭任命**　嵯峨天皇は809年から823年まで在位していました。藤原冬嗣は嵯峨天皇の時代に活躍した人物です。
* **ウ：承和の変と北家の優位確立**　承和の変は834年。その後の混乱を経て、藤原良房が北家の優位を確立しました。
* **ア：菅原道真の追放**　これは894年（寛平6年）の出来事です。


他の例題と同様に、歴史的順序に基づいて判断しました。  各出来事の正確な西暦年は必要ありませんが、おおよその時代背景を理解することで正しい順番を導き出すことができます。



## Gemmaを用いる場合

In [22]:
text = """
以下の問題に解答してください。

9世紀に活躍した人物に関係するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　藤原時平は，策謀を用いて菅原道真を政界から追放した。
イ　嵯峨天皇は，藤原冬嗣らを蔵人頭に任命した。
ウ　藤原良房は，承和の変後，藤原氏の中での北家の優位を確立した。



また、以下の問題を参考にしてください。

日本の近代化に関連するできごとについて述べた次のア～ウを年代の古い順に正しく並べよ。

ア　府知事・県令からなる地方官会議が設置された。
イ　廃藩置県が実施され，中央から府知事・県令が派遣される体制になった。
ウ　すべての藩主が，天皇に領地と領民を返還した。

解答: ウ→イ→ア

江戸幕府の北方での対外的な緊張について述べた次の文ア～ウを年代の古い順に正しく並べよ。

ア　レザノフが長崎に来航したが，幕府が冷淡な対応をしたため，ロシア船が樺太や択捉島を攻撃した。
イ　ゴローウニンが国後島に上陸し，幕府の役人に捕らえられ抑留された。
ウ　ラクスマンが根室に来航し，漂流民を届けるとともに通商を求めた。

解答: ウ→ア→イ

中居屋重兵衛の生涯の期間におこったできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　アヘン戦争がおこり，清がイギリスに敗北した。
イ　異国船打払令が出され，外国船を撃退することが命じられた。
ウ　桜田門外の変がおこり，大老の井伊直弼が暗殺された。

解答: イ→ア→ウ

加藤高明が外務大臣として提言を行ってから、内閣総理大臣となり演説を行うまでの時期のできごとについて述べた次のア～ウを，年代の古い順に正しく並べよ。

ア　朝鮮半島において，独立を求める大衆運動である三・一独立運動が展開された。
イ　関東大震災後の混乱のなかで，朝鮮人や中国人に対する殺傷事件がおきた。
ウ　日本政府が，袁世凱政府に対して二十一カ条の要求を突き付けた。

解答: ウ→ア→イ
"""

chat = [
    {
        "role": "user",
        "content": text},
]
prompt = tokenizer.apply_chat_template(chat, tokenize = False, add_generation_prompt = True)

# 1. モデルに入力するトークンを作成し、その長さを記憶
inputs = tokenizer.encode(prompt, add_special_tokens = False, return_tensors = "pt")
input_token_length = len(inputs[0])

# モデルに推論を実行させる
outputs = gemma.generate(input_ids=inputs.to(gemma.device), max_new_tokens = 512)

# 2. 全体の出力から、新しく生成された部分のトークンだけをスライス
new_tokens = outputs[0][input_token_length:]

# 3. 新しく生成された部分だけをデコードする
model_answer = tokenizer.decode(new_tokens, skip_special_tokens = True)

# モデルの解答だけを表示
print(model_answer)

9世紀に活躍した人物に関係するできごとについて述べたア～ウを年代の古い順に正しく並べようとした結果は、以下の通りとなります。

ウ→イ→ア

日本の近代化に関連するできごとについて述べたア～ウを年代の古い順に並べようとした結果は、以下の通りとなります。

ウ→ア→イ


## 42. 多肢選択問題の正解率

[JMMLU](https://github.com/nlp-waseda/JMMLU) のいずれかの科目を大規模言語モデルに解答させ、その正解率を求めよ。

In [3]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [4]:
import pandas as pd
import time

In [5]:
df = pd.read_csv('/content/drive/MyDrive/nlp100/lesson05/college_computer_science.csv', header = None, names = ['Question', 'A', 'B', 'C', 'D', 'Answer'])


In [31]:
corrects = 0
for index, row in df.iterrows():
  if index % 25 == 0:
    print(f'== {index + 1}問 開始 ==')
  question = row['Question']
  A = row['A']
  B = row['B']
  C = row['C']
  D = row['D']
  answer = row['Answer']

  prompt = f"""
  以下の問題を解いてください。回答は選択肢のアルファベット（A、B、C、D）のみを返してください。

  {question}
  A: {A}
  B: {B}
  C: {C}
  D: {D}
  """
  response = model.generate_content(prompt)
  # print(prompt)
  # print('\n\n', answer)
  # print('\n\n', response.text)

  if response.text == answer:
    corrects += 1
  time.sleep(5) # APIの利用枠を超えないため
  # break

print(f'正解率: {corrects / len(df)}')

== 1問 開始 ==
== 26問 開始 ==
== 51問 開始 ==
== 76問 開始 ==
正解率: 0.6161616161616161


## 43. 応答のバイアス

問題42において、実験設定を変化させると正解率が変化するかどうかを調べよ。実験設定の例としては、大規模言語モデルの温度パラメータ、プロンプト、多肢選択肢の順番、多肢選択肢の記号などが考えられる。

正解の選択肢を全てDに入れ替えて解答させる例。

In [6]:
# 実験設定の変化
for index, row in df.iterrows():
  question = row['Question']
  A = row['A']
  B = row['B']
  C = row['C']
  D = row['D']
  answer = row['Answer'] # A~Dのうちどれか
  if answer != 'D':
    if answer == 'A':
      df.at[index, 'A'] = D
      df.at[index, 'D'] = A
    elif answer == 'B':
      df.at[index, 'B'] = D
      df.at[index, 'D'] = B
    elif answer == 'C':
      df.at[index, 'C'] = D
      df.at[index, 'D'] = C
    df.at[index, 'Answer'] = 'D'

In [9]:
# 解答の保存
## 実際には、モデルの解答はプロンプトによって変化するので、RPDを節約するため、この勉強会でのみ保存する(だから、精度の高い入力構造を作成し、出力のズレが少なくなるようにする)
index_to_answer_by_model = dict()

In [13]:
# 多肢選択問題の正解率
corrects = 0
question_count = 0
try:
  for index, row in df.iterrows():
    if index % 25 == 0:
      print(f'== {index + 1}問 開始 ==')
    question_count += 1

    question = row['Question']
    A = row['A']
    B = row['B']
    C = row['C']
    D = row['D']
    answer = row['Answer']

    # 以前、問題を解答していれば、その解答を参照(RPD)の節約
    if index in index_to_answer_by_model:
      response_text = index_to_answer_by_model[index]
    else:
      prompt = f"""
      ## タスク
      以下の問題を解いてください。

      ## 入力
      {question}
      A: {A}
      B: {B}
      C: {C}
      D: {D}

      ## 出力形式
      回答は選択肢のアルファベット（A、B、C、D）のみを返してください。
      """
      response = model.generate_content(prompt)
      response_text = response.text
      # print(prompt)
      # print('\n\n', 'answer: ', answer)
      # print('\n\n', 'response.text: ', response_text)

      # この問題に対するmodelの解答を保存
      index_to_answer_by_model[index] = response.text
      time.sleep(5) # APIの利用枠を超えないため

    if response_text == answer:
      corrects += 1

    # break
  print(f'正解率: {corrects / question_count}')
except Exception as e: # もしエラーが発生しても無駄にならないように
  print(f'エラーが発生しました。処理を中断します。')
  print(f'エラーの詳細: {e}')
  print(f'回答した問題数: {question_count}')
  print(f'正解した問題数: {corrects}')
  print(f'回答した箇所までの正解率: {corrects / question_count}')

== 1問 開始 ==
== 26問 開始 ==
== 51問 開始 ==
== 76問 開始 ==
エラーが発生しました。処理を中断します。
エラーの詳細: 500 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint: An internal error has occurred. Please retry or report in https://developers.generativeai.google/guide/troubleshooting
回答した問題数: 86
正解した問題数: 52
回答した箇所までの正解率: 0.6046511627906976


ERROR:tornado.access:500 POST /v1beta/models/gemini-2.5-flash-lite:generateContent?%24alt=json%3Benum-encoding%3Dint (127.0.0.1) 6291.07ms


In [14]:
index_to_answer_by_model

{0: 'D',
 1: 'C',
 2: 'C1のクラスタ中心を計算します。C1は{(0,6), (6,0)}の2点を含みます。\nクラスタ中心のx座標の平均 = (0 + 6) / 2 = 3\nクラスタ中心のy座標の平均 = (6 + 0) / 2 = 3\nしたがって、C1のクラスタ中心は(3,3)です。\n\nC2のクラスタ中心を計算します。C2は{(2,2), (4,4), (6,6)}の3点を含みます。\nクラスタ中心のx座標の平均 = (2 + 4 + 6) / 3 = 12 / 3 = 4\nクラスタ中心のy座標の平均 = (2 + 4 + 6) / 3 = 12 / 3 = 4\nしたがって、C2のクラスタ中心は(4,4)です。\n\nC3のクラスタ中心を計算します。C3は{(5,5), (7,7)}の2点を含みます。\nクラスタ中心のx座標の平均 = (5 + 7) / 2 = 12 / 2 = 6\nクラスタ中心のy座標の平均 = (5 + 7) / 2 = 12 / 2 = 6\nしたがって、C3のクラスタ中心は(6,6)です。\n\n計算されたクラスタ中心は、C1: (3,3), C2: (4,4), C3: (6,6) です。\nこの結果と選択肢を比較すると、選択肢Dが一致します。\n\nD',
 3: 'D',
 4: 'B',
 5: 'D',
 6: 'A',
 7: 'A',
 8: 'D',
 9: 'D',
 10: 'D',
 11: 'C',
 12: 'D',
 13: 'B',
 14: 'D',
 15: 'D',
 16: 'A',
 17: 'D',
 18: 'D',
 19: 'D',
 20: 'D',
 21: 'D',
 22: 'B',
 23: 'D',
 24: 'D',
 25: 'D',
 26: 'D',
 27: 'D',
 28: 'D',
 29: 'D',
 30: 'D',
 31: 'D',
 32: 'A',
 33: 'D',
 34: 'A',
 35: 'C',
 36: 'A',
 37: 'A',
 38: 'D',
 39: 'D',
 40: 'D',
 41: 'A',
 42: 'B',
 43: 'D',
 44: 'D',
 45: 'D',
 46: 

## 44. 対話

以下の問いかけに対する応答を生成せよ。

> つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えました。東急大井町線の大井町方面の電車に乗り換えたとき、各駅停車に乗車すべきところ、間違えて急行に乗車してしまったことに気付きました。自由が丘の次の急行停車駅で降車し、反対方向の電車で一駅戻った駅がつばめちゃんの目的地でした。目的地の駅の名前を答えてください。

参考: [東急線・みなとみらい線路線案内](https://www.tokyu.co.jp/railway/station/map.html)

In [ ]:
prompt = """
## タスク
以下の問いかけに対する応答を生成せよ。

## 背景・文脈
- 東急東横線:
- 東急大井町線:
- 東急大井町線の急行停車駅:

## 入力
つばめちゃんは渋谷駅から東急東横線に乗り、自由が丘駅で乗り換えました。東急大井町線の大井町方面の電車に乗り換えたとき、各駅停車に乗車すべきところ、間違えて急行に乗車してしまったことに気付きました。自由が丘の次の急行停車駅で降車し、反対方向の電車で一駅戻った駅がつばめちゃんの目的地でした。目的地の駅の名前を答えてください。

## 出力形式
応答は文字列で返してください。
"""

response = model.generate_content(prompt)
print(response.text)

## 45. マルチターン対話

先ほどの応答に続けて、以下の追加の問いかけに対する応答を生成せよ。

> さらに、つばめちゃんが自由が丘駅で乗り換えたとき、先ほどとは反対方向の急行電車に間違って乗車してしまった場合を考えます。目的地の駅に向かうため、自由が丘の次の急行停車駅で降車した後、反対方向の各駅停車に乗車した場合、何駅先の駅で降りれば良いでしょうか？

## 46. 川柳の生成

適当なお題を設定し、川柳の案を10個作成せよ。

## 47. LLMによる評価

大規模言語モデルを評価者（ジャッジ）として、問題46の川柳の面白さを10段階で評価せよ。

## 48. LLMによる評価の頑健性

問題47で行ったLLMによるテキストの評価に関して、その頑健さ（脆弱さ）を調査せよ。最も単純な方法は、同じ評価を何回か繰り返した時のスコアの分散を調べることであろう。また、川柳の末尾に特定のメッセージを追加することで、評価スコアを恣意的に操作することも可能であろう。

## 49. トークン化

以下の文章（夏目漱石の『吾輩は猫である』の冒頭部分）のトークン数を計測せよ。

>　吾輩は猫である。名前はまだ無い。
>
>　どこで生れたかとんと見当がつかぬ。何でも薄暗いじめじめした所でニャーニャー泣いていた事だけは記憶している。吾輩はここで始めて人間というものを見た。しかもあとで聞くとそれは書生という人間中で一番獰悪な種族であったそうだ。この書生というのは時々我々を捕えて煮て食うという話である。しかしその当時は何という考もなかったから別段恐しいとも思わなかった。ただ彼の掌に載せられてスーと持ち上げられた時何だかフワフワした感じがあったばかりである。掌の上で少し落ちついて書生の顔を見たのがいわゆる人間というものの見始であろう。この時妙なものだと思った感じが今でも残っている。第一毛をもって装飾されべきはずの顔がつるつるしてまるで薬缶だ。その後猫にもだいぶ逢ったがこんな片輪には一度も出会わした事がない。のみならず顔の真中があまりに突起している。そうしてその穴の中から時々ぷうぷうと煙を吹く。どうも咽せぽくて実に弱った。これが人間の飲む煙草というものである事はようやくこの頃知った。
